# **🛩️ ATAS**

The ATAS (Airial Threat Assesment System) is an end-to-end machine learning engineering project that combines

- computer vision,
- structured machine learning,
- tactical scenario generation,
- and interactive visualization into a single pipeline.

**The project is divided into three major components:**

- Aircraft Classification Model
- ETA Regressor Model
- Hit Classification Model

---

**In this notebook**

# **⌚ ETA Regression Model**

This notebook focuses on building the ETA Regression Model using `scikit-learn` and `ensemble methods` to predict the minimum evasion time across `1,000,000` synthetic engagement scenarios.

## **1. Problem Definition**

Physics can tell you when a missile arrives that is simple geometry.

What it cannot tell you is **how much time the pilot actually needs to 
successfully evade it.**

Given the engagement conditions including aircraft states, missile 
specifications, and geometry, predict the **minimum time window** the pilot 
needs to initiate a successful evasion through maneuvering or deploying 
countermeasures, before the missile reaches a point where evasion becomes 
extremely difficult.

This minimum window depends on non-linear interactions between maneuverability, 
missile phase, enemy generation, and countermeasure state making it a genuine 
ML problem, not a physics calculation.

---

## **2. Data**

The data used here is synthetic, generated using real physics equations since 
real labeled engagement data does not exist publicly. This is standard practice; 
companies like Tesla and Waymo also train models on synthetic data.

**How the data was made:**
- Generated by `physics_generator.py` using real aircraft specs from `aircraft_metadata.csv`
- Stored as `synthetic_engagements.csv`

**Keys notes on Data**
- The data contains 16 columns, 1,000,000 rows
- **14 columns are input features:**
  
| Feature | Description |
|---|---|
| `launch_distance` | Distance from which the missile was fired (metres) |
| `remaining_distance` | Current distance of missile from your aircraft |
| `closure_rate` | Combined closing speed of missile + your aircraft (m/s) |
| `azimuth` | Horizontal angle of incoming threat (0°–360°) |
| `elevation` | Vertical angle of incoming threat (-90°–+90°) |
| `missile_phase` | 0 = boost, 1 = mid-course, 2 = terminal |
| `your_speed` | Your current airspeed (m/s) |
| `your_altitude` | Your current altitude (metres) |
| `your_maneuverability` | 0 = low, 1 = medium, 2 = high |
| `enemy_altitude` | Enemy aircraft altitude (metres) |
| `missile_speed` | Speed of incoming missile (m/s) |
| `missile_range` | Max effective range of missile (metres) |
| `enemy_generation` | Aircraft generation (3.5, 4, 4.5, 5) |
| `countermeasure_deployed` | 0 = not deployed, 1 = deployed |
  
- **2 columns are target labels:**
  
| Label | Description |
|---|---|
| `evasion_time` | Minimum time available to evade the missile (seconds) - **target for this model** |
| `hit` | Whether the missile hits after evasion (0 = miss, 1 = hit) - dropped here |

--- 

## **3. Evaluation**

The primary evaluation metric for this model is **MAE** (Mean Absolute Error) -
it tells us how many seconds off the prediction is on average, which is directly 
meaningful in a tactical context.

**All three metrics will be tracked:**

| Metric | Goal |
|---|---|
| MAE | As low as possible (seconds) |
| RMSE | As low as possible - penalises large errors harder than MAE |
| R² | > 0.90 |

---

> Note: The goal for all three metrics is to minimise error. A well-performing model should predict evasion time within a small margin of seconds large errors are tactically dangerous.

---

## **4. Features**

Some information about the data:

* We are dealing with structured tabular data so we will be using ensemble methods (Random Forest, XGBoost).
* The dataset contains `1,000,000` synthetic engagement scenarios across `101` aircraft types.
* All 14 input features are numerical - no text, no images, no missing values.
* Features span four groups:
   * **Engagement geometry:** `launch_distance`, `remaining_distance`, `closure_rate`, `azimuth`, `elevation`, `missile_phase`
   * **Your aircraft state:** `your_speed`, `your_altitude`, `your_maneuverability`
   * **Enemy aircraft state:** `enemy_altitude`
   * **Threat specs:** `missile_speed`, `missile_range`, `enemy_generation`, `countermeasure_deployed`
* The target variable is `evasion_time` (float, seconds) — the `hit` column is dropped.
* Data was generated deterministically from physics rules no noise, no class imbalance issues.
  
* Features like `azimuth`, `elevation`, and `countermeasure_deployed` introduce non-linear interactions that make this a genuine ML problem rather than a simple physics calculation.

---

## **5. Preparing the Tools**

We will be using the following libraries for this project:

* **NumPy** → numerical operations
* **Pandas** → loading and manipulating the synthetic engagement dataset
* **Matplotlib** → visualizing feature distributions and model performance
* **Scikit-learn** → model training, evaluation metrics, and train/test split
* **XGBoost** → gradient boosted trees (primary candidate)
* **Joblib** → saving and loading trained models

In [1]:
# Importing the Librabries
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import xgboost
import joblib

%matplotlib inline

In [2]:
# Use for naming the files
import datetime, pytz
ist = pytz.timezone("Asia/Kolkata")
datetime.datetime.now(ist).strftime('%Y-%m-%d_%H-%M-%S')

'2026-05-26_16-46-29'

## **6. Importing the data and preparing it for modelling**

In [3]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent.parent
PROJECT_ROOT

PosixPath('/mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project')

In [4]:
# loading the dataframe
aircraft_df = pd.read_csv(PROJECT_ROOT / "data/synthetic_engagements.csv",
                          low_memory=False)

# drop the hit coloumn
aircraft_df = aircraft_df.drop(columns="hit")
aircraft_df.head()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time
0,33895.443877,5709.789560,795.851952,110.969042,26.933223,2,191.657942,20767.395528,2,15722.748322,857,35000,4.0,1,6.708098
1,9775.992373,1963.280302,846.016340,91.346829,-24.938668,2,515.353333,24375.588350,2,10976.820086,857,35000,4.0,0,2.169777
2,24382.997906,7464.370311,829.119185,127.394632,47.364627,2,67.779862,21557.988948,0,6022.335489,857,35000,4.0,1,8.417591
3,28058.181279,17335.851081,868.412503,305.274787,81.459572,1,133.070710,7500.712655,2,18492.264381,857,35000,4.0,1,21.958961
4,22448.079023,9829.917985,517.446433,199.804139,32.414351,1,427.506675,4408.181588,1,11524.183399,857,35000,4.0,1,20.896675


In [5]:
# length of dataset
len(aircraft_df)

1000000

In [6]:
# Shape of dataset
aircraft_df.shape

(1000000, 15)

In [7]:
# Feature value distribution
aircraft_df.describe()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,65020.610752,32562.786912,1275.911046,179.937975,0.026018,1.009369,521.209036,14984.567041,1.001135,15009.517556,1273.921304,129450.708000,4.284310,0.500385,25.247238
std,74203.297027,46871.763979,468.190032,104.048678,52.038289,0.818479,265.998747,8666.432030,0.816686,8657.737189,370.271483,111055.688939,0.456488,0.500000,116.449639
min,500.024387,0.002412,0.075934,0.001255,-89.999940,0.000000,61.000864,0.008947,0.000000,0.036823,686.000000,8000.000000,3.500000,0.000000,0.000005
25%,9738.630388,3406.308603,906.492383,89.685133,-45.128021,0.000000,290.799123,7468.518845,0.000000,7532.535450,857.000000,35000.000000,4.000000,0.000000,3.459720
50%,37331.005140,13996.800173,1324.608648,179.910434,0.031418,1.000000,521.128004,14971.760950,1.000000,14998.923688,1372.000000,110000.000000,4.000000,1.000000,11.753448
75%,94126.351728,42326.795890,1569.077597,270.125866,45.145121,2.000000,751.299795,22489.688900,2.000000,22511.801778,1372.000000,160000.000000,4.500000,1.000000,32.672656
max,399998.390173,397823.101486,3033.120274,359.998621,89.999688,2.000000,982.998755,29999.961163,2.000000,29999.936049,2058.000000,400000.000000,5.000000,1.000000,69499.643685


In [8]:
aircraft_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   launch_distance          1000000 non-null  float64
 1   remaining_distance       1000000 non-null  float64
 2   closure_rate             1000000 non-null  float64
 3   azimuth                  1000000 non-null  float64
 4   elevation                1000000 non-null  float64
 5   missile_phase            1000000 non-null  int64  
 6   your_speed               1000000 non-null  float64
 7   your_altitude            1000000 non-null  float64
 8   your_maneuverability     1000000 non-null  int64  
 9   enemy_altitude           1000000 non-null  float64
 10  missile_speed            1000000 non-null  int64  
 11  missile_range            1000000 non-null  int64  
 12  enemy_generation         1000000 non-null  float64
 13  countermeasure_deployed  1000000 non-null  int64  
 14

In [9]:
# Checking for any null values
aircraft_df.isna().sum()

launch_distance            0
remaining_distance         0
closure_rate               0
azimuth                    0
elevation                  0
missile_phase              0
your_speed                 0
your_altitude              0
your_maneuverability       0
enemy_altitude             0
missile_speed              0
missile_range              0
enemy_generation           0
countermeasure_deployed    0
evasion_time               0
dtype: int64

In [10]:
# Looking at min and max values of evasion time
print(f"Min evasion time: {aircraft_df['evasion_time'].min()}")
print(f"Max evasion time: {aircraft_df['evasion_time'].max()}")  # ~19.3h

Min evasion time: 5.274148790829906e-06
Max evasion time: 69499.6436852845


Checking for extreme values in the target variable before training.
A realistic evasion window is between **1 second** (minimum reaction time) 
and **300 seconds** (5 minutes already unrealistically long).
Values outside this range are likely physics generator edge cases.

In [11]:
print((aircraft_df["evasion_time"] < 1).sum())
print((aircraft_df["evasion_time"] > 300).sum())

97645
595


- **595 rows** > **300s** - negligible impact
- **97,645 rows** < **1s** - this needs a decision

In [12]:
aircraft_df[aircraft_df["evasion_time"] < 1].describe()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time
count,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000,97645.000000
mean,14451.488294,567.217856,1134.683536,180.463469,0.049107,1.776005,513.584896,14989.083005,1.001854,15008.229712,1056.995514,68751.241743,4.090081,0.500783,0.475241
std,31082.639017,433.735741,443.942149,112.992610,52.283989,0.525874,265.686708,8665.810027,0.815798,8661.405303,371.733880,86404.886604,0.412108,0.500002,0.286996
min,500.024387,0.002412,10.602425,0.005499,-89.999026,0.000000,61.007723,0.478410,0.000000,0.348801,686.000000,8000.000000,3.500000,0.000000,0.000005
25%,1597.062550,231.808131,787.911910,73.825926,-45.485454,2.000000,284.210382,7434.568418,0.000000,7527.484779,750.000000,8000.000000,4.000000,0.000000,0.223685
50%,4005.755966,477.148287,1072.749878,180.876706,-0.095878,2.000000,506.844339,15019.860810,1.000000,14976.215582,857.000000,35000.000000,4.000000,1.000000,0.462742
75%,11103.466110,793.883753,1421.446517,286.521860,45.590001,2.000000,743.090612,22509.407764,2.000000,22540.529580,1372.000000,105000.000000,4.000000,1.000000,0.719451
max,398876.948563,3402.469398,3027.173477,359.998398,89.998819,2.000000,982.969035,29999.318347,2.000000,29999.936049,2058.000000,400000.000000,5.000000,1.000000,0.999999


### **Filtering Extreme Evasion Times**

Sub-second rows were investigated first before any filtering decision.
The pattern is clear these are not noise:

- `missile_phase` mean = 1.77 → heavily terminal phase
- `remaining_distance` mean = 567m → missile is already at close range
- `launch_distance` mean = 14,451m → short range engagements

This is the missile already in your face. Sub-second evasion time is 
physically correct in that scenario. These rows are kept. (97645 rows)

Rows with `evasion_time > 300s` are the actual edge cases - near-zero (closure rates producing unrealistic 19-hour windows)
closure rates producing unrealistic windows. **Dropping 595 rows out of 1,000,000.**

In [13]:
# Keeping the validated rows and dropping the extreme values
aircraft_df = aircraft_df[aircraft_df["evasion_time"] <= 300]
aircraft_df.head()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time
0,33895.443877,5709.789560,795.851952,110.969042,26.933223,2,191.657942,20767.395528,2,15722.748322,857,35000,4.0,1,6.708098
1,9775.992373,1963.280302,846.016340,91.346829,-24.938668,2,515.353333,24375.588350,2,10976.820086,857,35000,4.0,0,2.169777
2,24382.997906,7464.370311,829.119185,127.394632,47.364627,2,67.779862,21557.988948,0,6022.335489,857,35000,4.0,1,8.417591
3,28058.181279,17335.851081,868.412503,305.274787,81.459572,1,133.070710,7500.712655,2,18492.264381,857,35000,4.0,1,21.958961
4,22448.079023,9829.917985,517.446433,199.804139,32.414351,1,427.506675,4408.181588,1,11524.183399,857,35000,4.0,1,20.896675


In [14]:
len(aircraft_df)

999405

In [23]:
aircraft_df.dtypes

launch_distance            float64
remaining_distance         float64
closure_rate               float64
azimuth                    float64
elevation                  float64
missile_phase                int64
your_speed                 float64
your_altitude              float64
your_maneuverability         int64
enemy_altitude             float64
missile_speed                int64
missile_range                int64
enemy_generation           float64
countermeasure_deployed      int64
evasion_time               float64
dtype: object

## **7. Exploratory Data Analysis (EDA)**

**Before building a model, we need to understand the data.**

- Inspect distributions of all categorical columns
- Distribution plots for continuous features (launch_distance, closure_rate, evasion_time etc.)
- **Correlation heatmap:** which features are most correlated with evasion_time
- **evasion_time distribution plot:** check for skew, confirm the 300s filter worked
- **Pairplot or scatter plots:** relationships between key features and the target
- **Boxplots:** evasion_time split by missile_phase, enemy_generation, your_maneuverability
- **Outlier check:** anything suspicious still hiding in the continuous columns after the 300s filter

In [16]:
aircraft_df.describe()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,evasion_time
count,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.00000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000
mean,65013.755552,32542.830822,1276.555065,179.937820,0.025930,1.009734,520.993280,14984.533823,1.00115,15009.353638,1274.093497,129474.970608,4.284411,0.500388,24.486300
std,74170.064429,46805.403792,467.511980,104.078646,52.051722,0.818450,265.924147,8666.247323,0.81667,8657.791088,370.223841,111043.200468,0.456509,0.500000,32.324382
min,500.024387,0.002412,0.206626,0.001255,-89.999940,0.000000,61.000864,0.008947,0.00000,0.036823,686.000000,8000.000000,3.500000,0.000000,0.000005
25%,9739.140529,3404.931449,907.111492,89.627150,-45.155190,0.000000,290.638551,7468.446199,0.00000,7532.486649,857.000000,35000.000000,4.000000,0.000000,3.456294
50%,37354.954845,13994.035578,1324.890219,179.910579,0.032270,1.000000,520.832276,14971.837294,1.00000,14998.463827,1372.000000,110000.000000,4.000000,1.000000,11.737799
75%,94130.643570,42330.875735,1569.283500,270.177444,45.171235,2.000000,750.908502,22489.231268,2.00000,22511.722398,1372.000000,160000.000000,4.500000,1.000000,32.610377
max,399998.390173,397823.101486,3033.120274,359.998621,89.999688,2.000000,982.998755,29999.961163,2.00000,29999.936049,2058.000000,400000.000000,5.000000,1.000000,299.895722


In [18]:
aircraft_df["missile_phase"].value_counts()

missile_phase
2    339642
0    329914
1    329849
Name: count, dtype: int64

In [19]:
aircraft_df["your_maneuverability"].value_counts()

your_maneuverability
2    333851
1    332852
0    332702
Name: count, dtype: int64

In [20]:
aircraft_df["countermeasure_deployed"].value_counts()

countermeasure_deployed
1    500090
0    499315
Name: count, dtype: int64

In [22]:
aircraft_df["enemy_generation"].value_counts()

enemy_generation
4.0    489778
4.5    215634
5.0    215614
3.5     78379
Name: count, dtype: int64

### **Feature Engineering Decisions**

No encoding changes made. All categorical-looking columns left as numeric ordinal values.

Ordinal means the numbers have a meaningful order. 0 < 1 < 2 actually means something (low < medium < high). This is different from categories like colors where the numbers would be arbitrary.

| Column | Encoding | Reason |
|---|---|---|
| `missile_phase` | As-is (0/1/2) | Ordinal: phase 2 is genuinely more advanced than 0 |
| `your_maneuverability` | As-is (0/1/2) | Ordinal scale |
| `countermeasure_deployed` | As-is (0/1) | Already binary |
| `enemy_generation` | As-is (3.5/4/4.5/5) | Ordinal: tree models handle numeric thresholds fine |

No scaling applied. XGBoost, LightGBM, and RF split on numeric thresholds, scaling adds no value.

No imputation needed. Synthetic data, 0 nulls confirmed.

Distribution note: `missile_phase`, `your_maneuverability`, `countermeasure_deployed` are near-uniform by design (intentional in generator). `enemy_generation` skews toward Gen 4 (49%) which reflects metadata CSV composition, not a bug.